# Growing a mountain with a Neural Cellular Automaton

Trains a 16-channel NCA to grow the 🏔️ emoji from a single seed cell, and to keep it stable and regrow it after damage.
Based on [Mordvintsev et al., *Growing Neural Cellular Automata* (Distill, 2020)](https://distill.pub/2020/growing-ca/).
The model lives in `nca.py`; this notebook is the training loop.

Re-running the training cell continues from where it stopped. Interrupt the kernel to stop cleanly.

In [ ]:
import os, time, csv
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output

import nca

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
TARGET_PATH = "data/noto_1f3d4_snowcapped.png"   # or data/noto_26f0_mountain.png for ⛰️
TARGET_SIZE = 44        # emoji width in px, centered in the 64 px grid
POOL_SIZE = 1024
BATCH = 8
ITERS = int(os.environ.get("NCA_ITERS", 8000))
LR = 2e-3
LR_DROP_AT = 2000       # lr x0.1 after this many iterations
STEPS = (64, 96)        # CA steps per iteration, sampled uniformly
N_DAMAGED = 3           # pool samples per batch hit with a random erased circle (0 = growth + persistence only)
PLOT_EVERY = 100
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

torch.manual_seed(0)
print("device:", DEVICE, "| iterations:", ITERS)

In [ ]:
target = nca.load_target(TARGET_PATH, size=TARGET_SIZE).to(DEVICE)
plt.figure(figsize=(3, 3)); plt.imshow(nca.to_rgb(target)); plt.axis("off"); plt.title("target"); plt.show()

## Model, optimizer, sample pool

The pool holds 1024 grid states. Each batch draws 8 of them; the worst one is replaced by a fresh seed, and the
best few get a circle erased. After the update, the batch's final states go back into the pool, so the model is
trained on long-lived and damaged mountains, not only on growing from the seed.

In [ ]:
model = nca.CAModel().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[LR_DROP_AT], gamma=0.1)
pool = nca.make_seed(POOL_SIZE, device=DEVICE)
losses, it = [], 0
print(f"{sum(p.numel() for p in model.parameters()):,} trainable parameters")

In [ ]:
def per_sample_loss(x):
    return ((x[:, :4] - target) ** 2).mean(dim=(1, 2, 3))


def show(batch_before, batch_after):
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.5), gridspec_kw={"width_ratios": [3, 1]})
    row = lambda b: np.hstack(list(nca.to_rgb(b)))
    ax[0].imshow(np.vstack([row(batch_before), row(batch_after)])); ax[0].axis("off")
    ax[0].set_title(f"iteration {it}: pool samples before (top) and after (bottom)")
    ax[1].plot(losses, lw=0.5); ax[1].set_yscale("log"); ax[1].set_title(f"loss {np.mean(losses[-PLOT_EVERY:]):.5f}")
    plt.tight_layout(); plt.show()


log_path = CKPT_DIR / "train_log.csv"
t0 = time.time()
try:
    while it < ITERS:
        idx = torch.randperm(POOL_SIZE, device=DEVICE)[:BATCH]
        x = pool[idx].clone()
        with torch.no_grad():
            order = per_sample_loss(x).argsort(descending=True)   # worst first
            x, idx = x[order], idx[order]
            x[:1] = nca.make_seed(1, device=DEVICE)                 # worst -> fresh seed
            if N_DAMAGED:
                nca.damage(x[-N_DAMAGED:])                          # best few -> erase a circle
        before = x.detach()

        x = model.grow(x, int(torch.randint(STEPS[0], STEPS[1] + 1, ())))
        loss = per_sample_loss(x).mean()
        opt.zero_grad()
        loss.backward()
        for p in model.parameters():                                # per-parameter gradient normalization
            p.grad /= p.grad.norm() + 1e-8
        opt.step(); sched.step()

        pool[idx] = x.detach()
        losses.append(loss.item()); it += 1

        if it % PLOT_EVERY == 0:
            show(before, x.detach())
            print(f"{(time.time() - t0) / 60:.1f} min elapsed")
            with open(log_path, "a", newline="") as f:
                csv.writer(f).writerow([it, np.mean(losses[-PLOT_EVERY:]), sched.get_last_lr()[0], round(time.time() - t0)])
        if it % 1000 == 0:
            torch.save(model.state_dict(), CKPT_DIR / f"nca_{it:05d}.pth")
except KeyboardInterrupt:
    print(f"stopped at iteration {it}")

## Save

`.pth` for PyTorch, `.npz` for the numpy runtime in the Space.

In [ ]:
Path("models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "models/nca_mountain.pth")
nca.export_npz(model, "space/nca_mountain.npz")
print("saved models/nca_mountain.pth and space/nca_mountain.npz")

## Check: growth, persistence, regeneration

Grow from the seed for 600 steps (well past the 96 seen in training), erase the right half at step 300, and watch it come back.

In [ ]:
from PIL import Image

frames = []
with torch.no_grad():
    x = nca.make_seed(1, device=DEVICE)
    for step in range(600):
        if step == 300:
            x[..., nca.GRID // 2:] = 0
        x = model(x)
        if step % 2 == 0:
            frames.append((nca.to_rgb(x)[0] * 255).astype(np.uint8))

picks = [0, 20, 40, 75, 150, 299, 160, 175, 200, 299]   # frame indices (every 2 steps)
fig, ax = plt.subplots(1, len(picks), figsize=(2 * len(picks), 2.2))
for a, i in zip(ax, picks):
    a.imshow(frames[i]); a.axis("off"); a.set_title(f"step {2 * i}", fontsize=8)
plt.show()

Path("assets").mkdir(exist_ok=True)
gif = [Image.fromarray(f).resize((256, 256), Image.NEAREST) for f in frames]
gif[0].save("assets/grow.gif", save_all=True, append_images=gif[1:], duration=30, loop=0)
print("wrote assets/grow.gif")